# Lab 10 — Tokenization và Answer Span Alignment

Mạc Văn Tường — 23001951 — MAT3561E 4. Các bài tính tay và prediction nằm trong `calculations.md` và `prediction.md`.

In [1]:
from pathlib import Path
import os, sys, json
root=Path.cwd().resolve()
if (root/'qa_model.py').exists():
    root=root.parent
assert (root/'Lab10').exists(), 'Run from repository root or Lab10'
lab_dir=root/'Lab10'
sys.path.insert(0,str(lab_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from qa_model import (CONFIG, DEVICE, CHECKPOINT_DIR, seed_everything, load_splits,
    split_signature, get_tokenizer, encode_examples, align_span, normalize_answer,
    score_answer, example_score, evaluate_answers, KeywordBaseline, word_terms,
    sentence_ranges, train_or_restore, get_model, predict_examples, atomic_json)
seed_everything()
splits=load_splits(1000,200,200)
tokenizer=get_tokenizer()
print('Split sizes:',{k:len(v) for k,v in splits.items()})
print('Config:',CONFIG,'| device:',DEVICE)
print('Source SHA256:',split_signature(splits)['source_sha256'])

Split sizes: {'train': 1000, 'validation': 200, 'test': 200}
Config: {'model': 'albert/albert-base-v2', 'revision': '8e2f239c5f8a2c0f253781ca60135db913e5c80c', 'seed': 42, 'batch_size': 4, 'epochs': 2, 'learning_rate': 3e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'stride': 32, 'max_question_tokens': 64, 'max_answer_tokens': 64} | device: cpu
Source SHA256: {'train': '3527663986b8295af4f7fcdff1ba1ff3f72d07d61a20f487cb238a6ef92fd955', 'dev': '95aa6a52d5d6a735563366753ca50492a658031da74f301ac5238b03966972c9'}


## 10–12. Question/context tokenization và character offsets

In [2]:
question='What is the capital of France?'
context='The capital of France is Paris.'
answer_start=context.index('Paris');answer_end=answer_start+len('Paris')
encoding=tokenizer(question,context,return_offsets_mapping=True)
sequence_ids=encoding.sequence_ids()
start,end=align_span(encoding['offset_mapping'],sequence_ids,answer_start,answer_end,
                     encoding['input_ids'].index(tokenizer.cls_token_id))
display(pd.DataFrame(dict(token=encoding.tokens(),input_id=encoding['input_ids'],
    attention_mask=encoding['attention_mask'],token_type_id=encoding.get('token_type_ids', [None] * len(encoding['input_ids'])),
    sequence_id=sequence_ids,offset=encoding['offset_mapping'])))
print('Character span:',(answer_start,answer_end),'| token span inclusive:',(start,end))
print('Recovered:',context[encoding['offset_mapping'][start][0]:encoding['offset_mapping'][end][1]])
assert (answer_start,answer_end)==(25,30) and start==end==14


,token,input_id,attention_mask,token_type_id,sequence_id,offset
0,[CLS],2,1,None,NaN,"(0, 0)"
1,▁what,98,1,None,0.0,"(0, 4)"
2,▁is,25,1,None,0.0,"(5, 7)"
3,▁the,14,1,None,0.0,"(8, 11)"
4,▁capital,1057,1,None,0.0,"(12, 19)"
5,▁of,16,1,None,0.0,"(20, 22)"
6,▁france,714,1,None,0.0,"(23, 29)"
7,?,60,1,None,0.0,"(29, 30)"
8,[SEP],3,1,None,NaN,"(0, 0)"
9,▁the,14,1,None,1.0,"(0, 3)"


Character span: (25, 30) | token span inclusive: (14, 14)
Recovered: Paris


**Trả lời:** Character offsets đếm ký tự trong context gốc, word index đếm các từ của dãy minh họa, còn token index đếm vị trí trong input của model. `Paris` ở word 5 nhưng token 14 trong cặp input ALBERT. Dùng `sequence_ids()` để chỉ tìm span ở context, vì question cũng có offsets bắt đầu từ 0. ALBERT có `token_type_ids`; không giả định mọi Transformer đều cần trường này.

## 13 và 15. Sliding windows và overflow mapping

Minh họa đúng dạng `max_length=384, stride=128` trong đề. Các lần huấn luyện so sánh dùng stride 32 cố định cho cả 128/256/384; stride 128 không hợp lệ khi toàn bộ input chỉ dài 128 và còn phải chứa question/special tokens.

In [3]:
long_context=('A short notice describes the garden behind the building. '*80)+context
overflow=tokenizer(question,long_context,truncation='only_second',max_length=384,
    stride=128,return_overflowing_tokens=True,return_offsets_mapping=True)
display(pd.DataFrame(dict(window=range(len(overflow['input_ids'])),
    tokens=[len(x) for x in overflow['input_ids']],
    overflow_to_sample_mapping=overflow['overflow_to_sample_mapping'])))
sample=dict(id='toy-long',title='alignment example',question=question,context=long_context,
            answers=[dict(text='Paris',answer_start=long_context.index('Paris'))])
features,meta=encode_examples([sample],tokenizer,128)
display(pd.DataFrame([dict(window=i,example_index=m['example_index'],tokens=len(f['input_ids']),
    has_answer=m['has_answer'],start=f['start_positions'],end=f['end_positions']) for i,(f,m) in enumerate(zip(features,meta))]))
assert any(m['has_answer'] for m in meta) and any(not m['has_answer'] for m in meta)

,window,tokens,overflow_to_sample_mapping
0,0,384,0
1,1,384,0
2,2,325,0


,window,example_index,tokens,has_answer,start,end
0,0,0,128,False,0,0
1,1,0,128,False,0,0
2,2,0,128,False,0,0
3,3,0,128,False,0,0
4,4,0,128,False,0,0
5,5,0,128,False,0,0
6,6,0,128,False,0,0
7,7,0,128,False,0,0
8,8,0,128,True,126,126
9,9,0,43,True,40,40


**Trả lời:** Một question/context có thể tạo nhiều windows. `overflow_to_sample_mapping` nối mỗi window về sample gốc; `attention_mask` phân biệt padding. Offsets vẫn tham chiếu context gốc nên có thể ghép các candidate spans giữa windows. Window không chứa trọn đáp án dùng CLS làm target; điều này không có nghĩa question gốc không có đáp án.

## 16. Kiểm tra answer span alignment

In [4]:
# Synthetic offsets isolate the four required alignment edge cases.
offsets=[(0,0),(0,4),(5,9),(10,15),(0,0)]
seq=[None,1,1,1,None]
tests=[('answer starts inside token',1,9,(1,2)),
       ('answer ends inside token',0,8,(1,2)),
       ('partly truncated answer',5,18,(0,0)),
       ('answer outside window',20,24,(0,0))]
for name,a,b,expected in tests:
    actual=align_span(offsets,seq,a,b,0)
    assert actual==expected
    print(name,actual)
coverage=[]
for length in [128,256,384]:
    f,m=encode_examples(splits['train'],tokenizer,length)
    covered={x['example_id'] for x in m if x['has_answer']}
    clipped=sum(len(tokenizer(r['question'].strip(),add_special_tokens=False)['input_ids'])>CONFIG['max_question_tokens'] for r in splits['train'])
    coverage.append(dict(max_length=length,questions=len(splits['train']),features=len(f),
        questions_with_answer_window=len(covered),questions_without_answer_window=len(splits['train'])-len(covered),
        question_clipped=clipped))
display(pd.DataFrame(coverage))

answer starts inside token (1, 2)
answer ends inside token (1, 2)
partly truncated answer (0, 0)
answer outside window (0, 0)


,max_length,questions,features,questions_with_answer_window,questions_without_answer_window,question_clipped
0,128,1000,2116,1000,0,0
1,256,1000,1110,1000,0,0
2,384,1000,1013,1000,0,0


**Trả lời:** Khi biên gold nằm giữa một token, lấy token bao phủ ký tự đó; span khôi phục từ tokens có thể rộng hơn gold text. Nếu window không chứa đầy đủ hai biên, target là CLS. Training loss dùng reference đầu tiên; EM/F1 khi đánh giá lấy điểm cao nhất trên các references của question. Bảng coverage cho biết còn bao nhiêu question có ít nhất một window chứa đáp án, thay vì âm thầm bỏ các trường hợp khó.